In [ ]:
import importlib
import sys
from pathlib import Path

# Add mjosa_code root to path
mjosa_code_root = Path.cwd().parent  # Up to mjosa_code root
sys.path.insert(0, str(mjosa_code_root))

# Import from mjosa_code (NEW structure)
from utils.uhi import georef
from utils.common import config

importlib.reload(georef)  # RELOAD to get updated function!
from utils.uhi.georef import *

print(f"✅ Imports successful! mjosa_code root: {mjosa_code_root}")
print(f"✅ NEW FEATURES LOADED:")
print(f"   - Gaussian smoothing (smoothing_method='gaussian', gaussian_sigma=value)")
print(f"   - L2 normalization (normalization_method='l2')")

## Load UHI Data

In [ ]:
# Load transect 057
transect = load_transect(config.TRANSECT_057_OUTPUT)
transect.list_files()

# Select file from transect 057
cube = transect.select_files(["rad_uhi_20241029_115057_5"])
cube.describe()

In [ ]:
# Step 1: Apply illumination correction
cube.apply_illumination_correction_v2()

In [ ]:
# Step 2: Apply alignment adjustment
cube.adjust_uhi_alignment(dx=-0.05, dy=-3.0)

In [ ]:
cube.apply_wavelength_filter(wavelength_range=(490, 700))

In [ ]:
# Define analysis lines - ADJUSTED for track range (576-1566)
# Place lines within visible data range
line1 = [(410, 710), (940, 780)]
line2 = [(340, 1030), (340, 1210)]
line3 = [(250, 890), (250, 990)]
line4 = [(300, 1358), (300, 1458)]

lines = [line1, line2, line3, line4]
colors = ["red", "blue", "orange", "magenta"]
labels = ["Line 1", "Line 2", "Line 3", "Line 4"]

# Plot RGB georef - EXACT wavelengths: 677nm (R), 610nm (G), 490nm (B)
# plot_georef will find the CLOSEST available wavelength to these targets
cube.plot_georef(
    perimeter_line=lines,
    line_colors=colors,
    line_width=3,
    line_labels=labels,
    use_corrected=True,
    coordinate_system="NED",
    track_start=config.UHI_TRACK_RANGE[0],
    track_end=config.UHI_TRACK_RANGE[1],
    figsize=(50, 20),
    apply_alignment_shift=True,
)

# Intensity profiles for specific wavelength ranges (like dev10)
cube.plot_line_intensity_profile(
    perimeter_line=lines,
    use_average=True,
    moving_average_percent=20,
    show_markers=False,
    line_labels=labels,
    line_colors=colors,
    use_corrected=True,
    wavelength_range="all",  # Red bands (640-680nm) - change to 'green', 'blue', or (490,700)
    figsize=(10, 10),
)

In [ ]:
# Wavelength comparison with EXACT wavelengths and ACTUAL PHYSICAL COLORS
# Using the plot_wavelength_comparison function with single wavelength integers
cube.plot_wavelength_comparison(
    line=lines[0],  # Pick which line (0, 1, 2, or 3)
    wavelength_ranges=[
        677,  # Deep red (finds closest: 677.6nm) -> physical red color
        610,  # Orange (finds closest: 609.5nm) -> physical orange color
        490,  # Cyan (finds closest: 490.2nm) -> physical cyan color
    ],
    moving_average_percent=20,
    use_corrected=True,
    figsize=(10, 10),
)

# below i started testing different smoothing

---

---

---

---

---

---

---

---

---

---

---

---

---

---

---

---

## Test New Features: Gaussian Smoothing & L2 Normalization

**⚠️ IMPORTANT: Restart kernel and re-run cells 1-6 before running tests below!**

Testing the new parameters:
- `smoothing_method="gaussian"` with `gaussian_sigma`
- `normalization_method="l2"` for L2 normalization

New features added to both `plot_line_intensity_profile` and `plot_wavelength_comparison`!

In [ ]:
# Test 1: Gaussian smoothing with different sigma values
cube.plot_wavelength_comparison(
    line=lines[0],
    wavelength_ranges=[677, 610, 490],
    smoothing_method="gaussian",  # NEW: Gaussian smoothing
    gaussian_sigma=10.0,  # NEW: Sigma parameter
    use_corrected=True,
)
print("\n✅ Test 1 complete: Gaussian smoothing with sigma=2.0")

In [ ]:
# Test 2: Compare Gaussian (sigma=5.0) vs Moving Average
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))

# Left: Gaussian smoothing
plt.sca(ax1)
cube.plot_wavelength_comparison(
    line=lines[0],
    wavelength_ranges=[677, 610, 490],
    smoothing_method="gaussian",
    gaussian_sigma=5.0,
    use_corrected=True,
)
ax1.set_title("Gaussian Smoothing (σ=5.0)", fontsize=14, fontweight="bold")

# Right: Moving average
plt.sca(ax2)
cube.plot_wavelength_comparison(
    line=lines[0],
    wavelength_ranges=[677, 610, 490],
    smoothing_method="moving_average",
    moving_average_percent=20,
    use_corrected=True,
)
ax2.set_title("Moving Average (20%)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()
print("\n✅ Test 2 complete: Gaussian vs Moving Average comparison")

In [ ]:
# Test 3: L2 Normalization
cube.plot_wavelength_comparison(
    line=lines[0],
    wavelength_ranges=[677, 610, 490],
    smoothing_method="gaussian",
    gaussian_sigma=2.0,
    normalize_intensities=True,
    normalization_method="l2",  # NEW: L2 normalization (unit vector)
    use_corrected=True,
)
print("\n✅ Test 3 complete: L2 normalization")
print("⚠️  Note: L2 normalization makes the intensity vector have Euclidean norm = 1")

In [ ]:
# # Test 4: Compare all normalization methods
# fig, axes = plt.subplots(2, 2, figsize=(24, 16))

# normalization_methods = [
#     ("minmax", "MinMax [0-1]"),
#     ("zscore", "Z-Score"),
#     ("mean", "Mean-Relative"),
#     ("l2", "L2 (Unit Vector)"),
# ]

# for idx, (method, title) in enumerate(normalization_methods):
#     ax = axes[idx // 2, idx % 2]
#     plt.sca(ax)

#     cube.plot_wavelength_comparison(
#         line=lines[0],
#         wavelength_ranges=[677, 610, 490],
#         smoothing_method="gaussian",
#         gaussian_sigma=2.0,
#         normalize_intensities=True,
#         normalization_method=method,
#         use_corrected=True,
#     )
#     ax.set_title(f"{title} Normalization", fontsize=14, fontweight="bold")

# plt.tight_layout()
# plt.show()
# print("\n✅ Test 4 complete: All normalization methods compared")

In [ ]:
# Test 5: Line intensity profiles with Gaussian smoothing
cube.plot_line_intensity_profile(
    perimeter_line=lines,
    use_average=True,
    wavelength_range="all",
    smoothing_method="gaussian",  # NEW: Use Gaussian instead of moving average
    gaussian_sigma=3.0,  # NEW: Sigma parameter
    show_markers=False,
    line_labels=labels,
    line_colors=colors,
    use_corrected=True,
)
print("\n✅ Test 5 complete: Line intensity profiles with Gaussian smoothing")